# Story 3: within-eddy PV-regime transitions

Find long-lived eddies that spend sustained periods in both deep planetary-dominated water and shallower topographic-dominated water. These are the strongest explanatory cases because each eddy provides its own reference. CE ranking rewards following the rotating opposite-PV target; AE ranking rewards a clean planetary reference followed by weaker topographic directional coherence.

In [ ]:
from dataclasses import replace
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
HERE = Path.cwd().resolve()
ROOT = next((p for p in (HERE, *HERE.parents) if (p / 'seacofs_tilt_tools.py').exists()), None)
if ROOT is None: raise FileNotFoundError('Run from seacofs_eddy_tilt_analysis or a subfolder.')
for path in (ROOT, ROOT / 'case_studies'):
    if str(path) not in sys.path: sys.path.insert(0, str(path))
import seacofs_tilt_tools as tilt
from paper_case_study_tools import PaperCaseConfig, plot_paper_case, rank_regime_transition_cases, select_ranked_cases
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 40)

## Load and classify

A transition must contain sustained qualifying observations in both regimes and rotate the mean signed PV-gradient direction by at least 30 degrees. Mixed days remain visible in the time series but do not define either endpoint.

In [ ]:
config = PaperCaseConfig()
paths = tilt.Paths()
grid = tilt.load_grid(paths.grid, paths.z_r)
df_eddies, _ = tilt.load_tilt_tables(paths)
df_eddies = tilt.add_region_labels(df_eddies, grid)
df_eddies = tilt.add_pv_gradient_terms(df_eddies, grid, core_mean=True)
df_case, ranking = rank_regime_transition_cases(df_eddies, config)
display(ranking.groupby('response_type')['eligible'].agg(candidates='size', eligible='sum'))

## Ranked transitions

`pv_rotation_deg` is the change in the regime-mean signed PV-gradient bearing. `tilt_rotation_deg` is the corresponding change in mean tilt bearing. `rotation_coupling_error_deg` measures whether those rotations agree after applying the polarity-specific expected target.

In [ ]:
columns = ['Eddy','Cyc','Region','response_type','case_score','lifetime_days','planetary_observations','topographic_observations','longest_planetary_run','longest_topographic_run','regime_balance','planetary_preference_error_deg','topographic_preference_error_deg','topographic_relative_resultant','pv_rotation_deg','tilt_rotation_deg','rotation_coupling_error_deg','response_quality','planetary_tilt_km','topographic_tilt_km']
for response in ranking.response_type.unique():
    print(f'\n{response}')
    display(ranking.loc[(ranking.response_type.eq(response)) & ranking.eligible, columns].head(20).round(3))

In [ ]:
selected = select_ranked_cases(ranking, 'response_type', n_per_group=3)
selected

## Candidate transition figures

Blue and orange shading mark the strict planetary and topographic endpoints. Inspect the intervening mixed period: the strongest example should show a spatially coherent shelf/slope encounter and a temporally connected evolution of PV-gradient direction and tilt.

In [ ]:
for response, eddies in selected.items():
    for eddy_id in eddies:
        track = df_case[df_case.Eddy.eq(eddy_id)]
        plot_paper_case(track, grid, config=config, title=f'{response} - eddy {eddy_id}')
        plt.show()

## Within-eddy endpoint comparison

This descriptive table compares strict endpoints inside each selected eddy. It is not a test with independent daily replicates.

In [ ]:
selected_ids = [eddy for values in selected.values() for eddy in values]
rows = []
for eddy_id in selected_ids:
    part = df_case[df_case.Eddy.eq(eddy_id)]
    for regime, mask in [('planetary', part.planetary_regime), ('topographic', part.topographic_regime)]:
        use = part[mask & part.direction_valid]
        rows.append({'Eddy':eddy_id,'Cyc':part.Cyc.iloc[0],'regime':regime,'observations':len(use),'median_preference_error_deg':use.preference_error_deg.median(),'median_tilt_km':use.TiltDis.median(),'median_depth_m':use.h.median(),'median_Ro':use.Ro.median(),'median_ratio':use.topo_plan_ratio_smooth.median()})
display(pd.DataFrame(rows).sort_values(['Cyc','Eddy','regime']).round(3))

## Threshold sensitivity

Check whether the selected eddies remain eligible when dominance is tightened from 2:1 to 4:1. Failure means the contrast depends on moderately separated regimes; it does not automatically invalidate the example.

In [ ]:
strict = replace(config, dominance_factor=4.0)
_, strict_ranking = rank_regime_transition_cases(df_eddies, strict)
strict_pairs = set(zip(strict_ranking.loc[strict_ranking.eligible,'response_type'], strict_ranking.loc[strict_ranking.eligible,'Eddy']))
sensitivity = pd.DataFrame([(group,eddy,(group,eddy) in strict_pairs) for group,eddies in selected.items() for eddy in eddies], columns=['response_type','Eddy','eligible_at_4_to_1'])
display(sensitivity)

## Interpretation guardrails

Prioritize a clean geographic transition, continuous detection, sufficient tilt magnitude, and sustained endpoints. For CEs, show whether tilt rotates with the opposite-PV target. For AEs, show a stable planetary relationship followed by a loss of coherence rather than implying that every topographic AE adopts the same new direction. Use the static notebooks as population and endpoint context.